# Pydantic model to structure output

TODO: skapa en agent som ska simulera en IT-anställd

In [9]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
system_prompt="""
You are an HR expert within IT field in sweden within data science, data engineering, machine lerning, AI engineerin. You will simulate IT employees.

Fields to include: 
    
Name
Age
Gender
Job_title
Salary in SEK per month
""",)

prompt = "Simulate two employees"

result = await agent.run(prompt)

result

AgentRunResult(output="Here are two simulated IT employee profiles in Sweden:\n\n---\n\n### **Employee 1**  \n- **Name:** Astrid Mallette  \n- **Age:** 28 years  \n- **Gender:** Female  \n- **Job_title:** Senior Developer  \n- **Salary:** 92,000 SEK/month  \n\n---\n\n### **Employee 2**  \n- **Name:** Lars Eriksson  \n- **Age:** 32 years  \n- **Gender:** Male  \n- **Job_title:** Systems Analyst  \n- **Salary:** 115,000 SEK/month  \n\n---\n\nThese profiles reflect typical roles, genders, and salary ranges for IT professionals in Sweden. Let me know if you'd like adjustments!")

In [10]:
print(result.output)

Here are two simulated IT employee profiles in Sweden:

---

### **Employee 1**  
- **Name:** Astrid Mallette  
- **Age:** 28 years  
- **Gender:** Female  
- **Job_title:** Senior Developer  
- **Salary:** 92,000 SEK/month  

---

### **Employee 2**  
- **Name:** Lars Eriksson  
- **Age:** 32 years  
- **Gender:** Male  
- **Job_title:** Systems Analyst  
- **Salary:** 115,000 SEK/month  

---

These profiles reflect typical roles, genders, and salary ranges for IT professionals in Sweden. Let me know if you'd like adjustments!


In [11]:
with open("simulated_emlpoyees", "w") as file:
    file.write(result.output)

## Get more structured output

issue with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repeatable structure

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class EmployeeModel(BaseModel):
    name: str = Field(description="Mostly swedish names, but could be foreign names as well")
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(gte=30_000, lte=50_000, description="salary should be between 30k and 50k, the higher experience level, the higher salary") 

load_dotenv()

agent = Agent(
    "openrouter:arcee-ai/trinity-mini:free",
system_prompt="""
You are an HR expert within IT field in sweden within data science, data engineering, machine lerning, AI engineerin. You will simulate IT employees.
""",)

result = await agent.run("Give me 3 employees", output_type=EmployeeModel)

C:\Users\norel\AppData\Local\Temp\ipykernel_25800\1629398058.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(gte=30_000, lte=50_000, description="salary should be between 30k and 50k, the higher experience level, the higher salary")


In [21]:
result.output

EmployeeModel(name='Anna Svensson', age=30, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=45000)

In [22]:
result.output.salary

45000

In [23]:
result = await agent.run("Give me 3 employees", output_type=list[EmployeeModel])

In [24]:
result.output

[EmployeeModel(name='Anna Svensson', age=35, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=45000),
 EmployeeModel(name='Lars Johansson', age=40, gender='Male', experience_level='Senior', job_title='Machine Learning Engineer', salary=48000),
 EmployeeModel(name='Maria Andersson', age=30, gender='Female', experience_level='Entry', job_title='Data Engineer', salary=32000)]

In [26]:
result.output[0]

EmployeeModel(name='Anna Svensson', age=35, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=45000)

In [25]:
result.output[0].model_dump()

{'name': 'Anna Svensson',
 'age': 35,
 'gender': 'Female',
 'experience_level': 'Mid level',
 'job_title': 'Data Scientist',
 'salary': 45000}

### TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on this list

In [39]:
employees = result.output
completed_list = []
for employee in employees:
    list_of_employees = employee.model_dump()
    completed_list.append(list_of_employees)
completed_list

[{'name': 'Anna Svensson',
  'age': 35,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Data Scientist',
  'salary': 45000},
 {'name': 'Lars Johansson',
  'age': 40,
  'gender': 'Male',
  'experience_level': 'Senior',
  'job_title': 'Machine Learning Engineer',
  'salary': 48000},
 {'name': 'Maria Andersson',
  'age': 30,
  'gender': 'Female',
  'experience_level': 'Entry',
  'job_title': 'Data Engineer',
  'salary': 32000}]

More simple code

In [42]:
employee_list = [employee.model_dump() for employee in employees]
employee_list

[{'name': 'Anna Svensson',
  'age': 35,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Data Scientist',
  'salary': 45000},
 {'name': 'Lars Johansson',
  'age': 40,
  'gender': 'Male',
  'experience_level': 'Senior',
  'job_title': 'Machine Learning Engineer',
  'salary': 48000},
 {'name': 'Maria Andersson',
  'age': 30,
  'gender': 'Female',
  'experience_level': 'Entry',
  'job_title': 'Data Engineer',
  'salary': 32000}]

In [44]:
import pandas as pd
df = pd.DataFrame([employee.model_dump() for employee in employees])
df

,name,age,gender,experience_level,job_title,salary
0,Anna Svensson,35,Female,Mid level,Data Scientist,45000
1,Lars Johansson,40,Male,Senior,Machine Learning Engineer,48000
2,Maria Andersson,30,Female,Entry,Data Engineer,32000


In [ ]:
df = pd.DataFrame(completed_list)
df

,name,age,gender,experience_level,job_title,salary
0,Anna Svensson,35,Female,Mid level,Data Scientist,45000
1,Lars Johansson,40,Male,Senior,Machine Learning Engineer,48000
2,Maria Andersson,30,Female,Entry,Data Engineer,32000


In [46]:
df["salary"].mean()

np.float64(41666.666666666664)